In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd "drive/MyDrive/reasoning"

/content/drive/MyDrive/reasoning


In [3]:
from helpers.qwen3 import download_qwen3_small, Qwen3Tokenizer, Qwen3Model, QWEN_CONFIG_06_B

def load_model_and_tokenizer(which_model, device, use_compile, local_dir="qwen3"):
    if which_model == "base":
        download_qwen3_small(kind="base", tokenizer_only=False, out_dir="math")
        tokenizer_path = "math/tokenizer-base.json"
        model_path = "math/qwen3-0.6B-base.pth"
        tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)
    elif which_model == "reasoning":
        download_qwen3_small(kind="reasoning", tokenizer_only="False", out_dir="math")
        tokenizer_path = "math/tokenizer-reasoning.json"
        model_path = "math/qwen3-0.6B-reasoning.pth"
        tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path, apply_chat_template=True, add_generation_prompt=True, add_thinking=True,)
    else:
        raise ValueError(f"Invalid choice")
    
    model = Qwen3Model(QWEN_CONFIG_06_B)
    model.load_state_dict(torch.load(model_path))
    model.to(device)
    
    if use_compile:
        torch._dynamo.config.allow_unspec_int_on_nn_module = True
        model = torch.compile(model)
        
    return model, tokenizer

In [4]:
import torch
device = torch.device("cuda")
model, tokenizer = load_model_and_tokenizer(which_model="base", device=device, use_compile=False)

✓ math/qwen3-0.6B-base.pth already up-to-date


In [5]:
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "Answer the question and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Question:\n{prompt}\n\nAnswer:"
    )
    return template

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

In [6]:
import torch
from helpers.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_stream_cache(model, token_ids, max_new_tokens, eos_token_id=None):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    
    out = model(token_ids, cache=cache)[:, -1]    
    for step in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)
        
        if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
            break
        
        yield next_token
        if step + 1 < max_new_tokens:
            out = model(next_token, cache=cache)[:, -1]

In [7]:
def generate_text_stream_concat_flex(model, tokenizer, prompt, device, max_new_tokens, verbose=False, generate_func=None, **generate_kwargs,):
    if generate_func is None:
        generate_func = generate_text_basic_stream_cache
        
    input_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
    generated_ids = []
    for token in generate_func(model=model, token_ids=input_ids, max_new_tokens=max_new_tokens, eos_token_id=tokenizer.eos_token_id, **generate_kwargs,):
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id.item())
        if verbose:
            print(tokenizer.decode(next_token_id.tolist()), end="", flush=True)
    return tokenizer.decode(generated_ids)

In [8]:
def scale_logits_by_temperature(logits, temperature):
    if temperature <= 0:
        raise ValueError("Temperature must be positive")
    return logits / temperature

In [9]:
def top_p_filter(probas, top_p):
    if top_p is None or top_p >= 1.0:
        return probas
    
    sorted_probas, sorted_idx = torch.sort(probas, dim=1, descending=True)
    cumprobas = torch.cumsum(sorted_probas, dim=1)
    prefix = cumprobas - sorted_probas
    
    keep = prefix < top_p
    keep[:, 0] = True
    keep_sorted = torch.where(keep, sorted_probas, torch.zeros_like(sorted_probas))
    
    filtered = torch.zeros_like(probas).scatter(1, sorted_idx, keep_sorted)
    denom = torch.sum(filtered, dim=1, keepdim=True).clamp_min(1e-12)
    return filtered / denom

In [10]:
from helpers.qwen3 import KVCache

@torch.inference_mode()
def generate_text_top_p_stream_cache(model, token_ids, max_new_tokens, eos_token_id=None, temperature=0., top_p=None):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    
    out = model(token_ids, cache=cache)[:, -1]    
    for step in range(max_new_tokens):
        orig_device = token_ids.device
        if temperature is None or temperature == 0.0:
            next_token = torch.argmax(out, dim=-1, keepdim=True)
        else:
            logits = scale_logits_by_temperature(out, temperature)
            probas = torch.softmax(logits, dim=-1)
            probas = top_p_filter(probas, top_p)
            next_token = torch.multinomial(probas.cpu(), num_samples=1)
            next_token = next_token.to(orig_device)
        
        if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
            break
        
        yield next_token
        if step + 1 < max_new_tokens:
            out = model(next_token, cache=cache)[:, -1]

In [11]:
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device, 
    max_new_tokens=2048, verbose=True, 
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

 \boxed{x = 37}

In [12]:
import json
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)
    with local_path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    return data

In [13]:
math_train = load_math_train()
print("Dataset size:", len(math_train))

Dataset size: 12000


In [14]:
from pprint import pprint

pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

In [15]:
@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device, 
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(tokenizer.encode(prompt), device=device)
    
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]
    
    generated = []
    for step in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature
            
        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(probas.cpu(), num_samples=1).to(device)
        
        token_id = next_token.item()
        generated.append(token_id)
        
        if tokenizer.eos_token_id is not None and token_id == tokenizer.eos_token_id:
            break
        
        if step + 1 < max_new_tokens:
            logits = model(next_token, cache=cache)[:, -1]
            
    full_token_ids = torch.cat([input_ids, torch.tensor(generated, device=device, dtype=input_ids.dtype),])
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [16]:
token_ids, prompt_len, answer_text = sample_response(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    device=device,
    max_new_tokens=512,
    temperature=0.9,
    top_p=0.9,
)

print(answer_text)

 \boxed{20}<|endoftext|>


In [17]:
from helpers.ch03 import extract_final_candidate, grade_answer

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(answer_text, fallback=None)
    if not extracted:
        return 0.0
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [18]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]
rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

Answer: '\\boxed{83}'
Reward: 1.0

Answer: 'The correct answer is \\boxed{83}'
Reward: 1.0

Answer: 'The final answer is 83'
Reward: 0.0

Answer: 'We get \\boxed{38}'
Reward: 0.0



In [19]:
rewards = torch.tensor(rollout_rewards, device=device)
print(rewards)

tensor([1., 1., 0., 0.], device='cuda:0')


In [20]:
advantages = (rewards - rewards.mean() / (rewards.std() + 1e-4))
print(advantages)

tensor([ 0.1341,  0.1341, -0.8659, -0.8659], device='cuda:0')


In [21]:
@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)
    
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)
    
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1
    
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start+1: end+1]
    next_token_logps = logprobs[t_idx, next_tokens]
    
    return torch.mean(next_token_logps).item()

In [22]:
avg_logprob_val = avg_logprob_answer(
    model, tokenizer, prompt=prompt,
    answer=answer_text, device=device
)
print(avg_logprob_val)

-0.6953125


In [23]:
sequence_logprob_val = avg_logprob_val * (len(tokenizer.encode(answer_text)))
print(sequence_logprob_val)

-4.8671875


In [24]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    
    start = prompt_len - 1
    end = token_ids.shape[0] - 1
    
    t_idx = torch.arange(start, end, device=device)
    next_tokens = token_ids[start+1: end+1]
    next_token_logps = logprobs[t_idx, next_tokens]
    
    return torch.sum(next_token_logps)

print(sequence_logprob_draft(model, token_ids, prompt_len))

tensor(-4.8767, device='cuda:0', grad_fn=<SumBackward0>)


In [25]:
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(1, token_ids[1:].unsqueeze(-1)).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

tensor(-4.8767, device='cuda:0', grad_fn=<SumBackward0>)


In [26]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]
rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(model=model, token_ids=torch.tensor(token_ids, device=device), prompt_len=prompt_len)
    
    print(f"Answer: {text}")
    print(f"Logprob: {logprob.item():.4f}\n")
    
    rollout_logps.append(logprob)

Answer: \boxed{83}
Logprob: -8.1801

Answer: The correct answer is \boxed{83}
Logprob: -19.8252

Answer: The final answer is 83
Logprob: -16.2867

Answer: We get \boxed{38}
Logprob: -23.4763



In [27]:
logps = torch.stack(rollout_logps)
print(logps)

tensor([ -8.1801, -19.8252, -16.2867, -23.4763], device='cuda:0',
       grad_fn=<StackBackward0>)


In [28]:
pg_loss = -(advantages.detach() * logps).mean()
print(pg_loss)

tensor(-7.6684, device='cuda:0', grad_fn=<NegBackward0>)


In [29]:
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])
    
    was_training = model.training
    model.eval()
    
    for _ in range(num_rollouts):
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        reward = reward_rlvr(text, example["answer"])
        logp = sequence_logprob(model, token_ids, prompt_len)
        
        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append({"text": text, "reward": reward, "gen_len": token_ids.numel() - prompt_len,})
        
    if was_training:
        model.train()
        
    rewards = torch.tensor(roll_rewards, device=device)
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
    logps = torch.stack(roll_logps)
    
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss
    
    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

In [30]:
stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

{'advantages': [0.0, 0.0],
 'loss': -0.0,
 'loss_tensor': tensor(-0., device='cuda:0', grad_fn=<NegBackward0>),
 'pg_loss': -0.0,
 'rewards': [0.0, 0.0],
 'samples': [{'gen_len': 131,
              'reward': 0.0,
              'text': " Let's assume that Sam worked for x days.\n"
                      'For each day he did not work, he lost $30.\n'
                      'So, for x days he lost $30x.\n'
                      'The total amount he received is $660, so we can write '
                      'the equation as:\n'
                      '$60x - 30x = $660\n'
                      '$30x = $660\n'
                      'Divide both sides of the equation by $30 to solve for '
                      'x:\n'
                      '$x = $660 / $30\n'
                      '$x = 22\n'
                      'Sam did not work for 22 days.\n'
                      'The answer is: 22<|endoftext|>'},
             {'gen_len': 6, 'reward': 0.0, 'text': ' \\boxed{8}<|endoftext|>'}]}


In [31]:
import time

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,

):
    if steps is None:
        steps = len(math_data)
        
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0
    
    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        for step in range(steps):
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            stats["loss_tensor"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(sample["gen_len"] for sample in stats["samples"])
            avg_response_len = (step_tokens / len(stats["samples"]) if stats["samples"] else 0.0)
            append_csv_metrics(csv_log_path, current_step, steps, stats["loss"], reward_avg, avg_response_len,)

            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            if current_step % 10 == 0:
                print(f"[Step {current_step}] sample outputs")
                for i, sample in enumerate(stats["samples"][:3]):
                    text = sample["text"].replace("\n", "\\n")
                    print(
                        f"  {i+1}) reward={sample['reward']:.3f} "
                        f"len={sample['gen_len']}: {text}"
                    )
                print()

            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model


def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    step_idx,
    total_steps,
    loss,
    reward_avg,
    avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

In [32]:
device = torch.device("cuda")
model.to(device)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=4,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
    csv_log_path="train_rlvr_grpo_metrics.csv"
)

[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=132.8
[Step 2/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=133.8
[Step 3/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=23.5
[Step 4/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=4.5
[Step 5/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=132.2
Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005.pth
[Step 6/50] loss=6.3326 reward_avg=0.250 avg_resp_len=236.0
[Step 7/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=266.2
[Step 8/50] loss=4.4463 reward_avg=0.750 avg_resp_len=177.5


OutOfMemoryError: CUDA out of memory. Tried to allocate 22.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 5.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.22 GiB is allocated by PyTorch, and 203.58 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)